================================================================
FILE : predict.py
WHAT THIS FILE DOES :
  Step 1 - Load trained Random Forest model
  Step 2 - Load input CSV file
  Step 3 - Show the input data on screen
  Step 4 - Run predictions on every file
  Step 5 - Show all predictions below the code
  Step 6 - Save predictions to results/predictions.csv

HOW TO RUN :  python predict.py
RUN AFTER  :  train.py  and  evaluate.py

TO PREDICT YOUR OWN FILES:
  Change  input_file = "data/test.csv"
  To      input_file = "your_file.csv"
================================================================

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np

In [ ]:
os.makedirs("results", exist_ok=True)

In [ ]:
# ----------------------------------------------------------------
# SETTINGS
# input_file  = CSV file you want to predict on
# threshold   = score >= this value → predicted as Malicious
#               0.5 means 50% confidence needed to call it malicious
# ----------------------------------------------------------------
input_file = "data/test.csv"
threshold  = 0.5

In [ ]:
print("=" * 65)
print("        PREDICTION  —  Step 4 of 4")
print("=" * 65)
print(f"        Input file : {input_file}")
print(f"        Threshold  : {threshold}  (score >= {threshold} = Malicious)")
print("=" * 65)

================================================================
STEP 1 : LOAD TRAINED MODEL
================================================================
pickle.load unfreezes the saved model from train.py

In [ ]:
print("\n[STEP 1]  Loading trained model...")

In [ ]:
with open("results/random_forest.pkl",  "rb") as f: rf            = pickle.load(f)
with open("results/feature_names.pkl",  "rb") as f: feature_names = pickle.load(f)

In [ ]:
print(f"         Random Forest model loaded")
print(f"         Trees in model     :  {rf.n_estimators}")
print(f"         Features expected  :  {len(feature_names)}")

================================================================
STEP 2 : LOAD INPUT FILE
================================================================

In [ ]:
print(f"\n[STEP 2]  Loading input file:  {input_file}")

In [ ]:
df = pd.read_csv(input_file)

In [ ]:
# Save actual labels if they exist (for accuracy check)
true_labels = None
if "label" in df.columns:
    true_labels = df["label"].values
    df = df.drop(columns=["label"])

In [ ]:
# Make sure columns match training features exactly
df = df.reindex(columns=feature_names, fill_value=0)
df = df.fillna(0)

In [ ]:
print(f"         Files to predict  :  {len(df)}")
print(f"         Features          :  {df.shape[1]}")

================================================================
STEP 3 : SHOW INPUT DATA
================================================================

In [ ]:
print("\n" + "=" * 65)
print("  INPUT DATA  —  Overview")
print("=" * 65)
print(f"  Total files  :  {len(df)}")
print(f"  Features     :  {df.shape[1]}")
print()
print("  First 5 rows of input data (first 8 features):")
print()
print(df.iloc[:, :8].head(5).to_string(index=True))

In [ ]:
if true_labels is not None:
    print()
    print("  Actual labels (from file):")
    print(f"    Benign    (0)  :  {(true_labels==0).sum()}")
    print(f"    Malicious (1)  :  {(true_labels==1).sum()}")

================================================================
STEP 4 : RUN PREDICTIONS
================================================================
predict_proba gives a probability between 0 and 1 for each file

  Score close to 1.0  = model is very confident → MALICIOUS
  Score close to 0.0  = model is very confident → Benign
  Score around 0.5    = model is uncertain

We compare score to threshold (default 0.5):
  score >= 0.5  → predict Malicious (1)
  score <  0.5  → predict Benign (0)

In [ ]:
print("\n[STEP 4]  Running predictions...")

In [ ]:
rf_proba = rf.predict_proba(df)[:, 1]    # probability 0 to 1
rf_pred  = (rf_proba >= threshold).astype(int)  # 0 or 1

In [ ]:
print(f"         Predictions complete for {len(df)} files")

================================================================
STEP 5 : SHOW PREDICTIONS BELOW CODE
================================================================

In [ ]:
# Count results
total_files    = len(df)
flagged_mal    = int(rf_pred.sum())
flagged_benign = total_files - flagged_mal
high_risk      = (rf_proba >= 0.9).sum()
medium_risk    = ((rf_proba >= 0.5) & (rf_proba < 0.9)).sum()
low_risk       = (rf_proba < 0.5).sum()

In [ ]:
print("\n" + "=" * 65)
print("  PREDICTION RESULTS — ALL FILES")
print("=" * 65)
print()

In [ ]:
# Full table — ALL files
print(f"  {'Nr':>5}  {'Score':>8}  {'Prediction':>11}  {'Confidence':>12}", end="")
if true_labels is not None:
    print(f"  {'Actual':>10}  {'Correct?':>8}", end="")
print()
print("  " + "-" * (55 if true_labels is None else 78))

In [ ]:
for i in range(len(df)):
    score      = rf_proba[i]
    prediction = "MALICIOUS" if rf_pred[i] == 1 else "Benign"

    # Confidence label based on score
    if score >= 0.9:
        confidence = "VERY HIGH"
    elif score >= 0.7:
        confidence = "HIGH"
    elif score >= 0.5:
        confidence = "MEDIUM"
    elif score >= 0.3:
        confidence = "LOW"
    else:
        confidence = "VERY LOW"

    row = f"  {i+1:>5}  {score:>8.4f}  {prediction:>11}  {confidence:>12}"

    if true_labels is not None:
        actual  = "Malicious" if true_labels[i] == 1 else "Benign"
        correct = "YES" if rf_pred[i] == true_labels[i] else "NO ←"
        row    += f"  {actual:>10}  {correct:>8}"

    print(row)

    # Only show first 50 files to avoid too long output
    if i == 49 and len(df) > 50:
        print(f"\n  ... showing first 50 of {len(df)} files ...")
        print(f"  See results/predictions.csv for ALL {len(df)} predictions")
        break

In [ ]:
print("  " + "-" * (55 if true_labels is None else 78))

================================================================
STEP 6 : SAVE PREDICTIONS
================================================================

In [ ]:
out_df = pd.DataFrame({
    "File_Number" : range(1, len(df)+1),
    "RF_Score"    : rf_proba.round(4),
    "Prediction"  : ["Malicious" if p==1 else "Benign" for p in rf_pred],
    "Confidence"  : ["VERY HIGH" if s>=0.9 else
                     "HIGH"      if s>=0.7 else
                     "MEDIUM"    if s>=0.5 else
                     "LOW"       if s>=0.3 else
                     "VERY LOW"  for s in rf_proba],
})

In [ ]:
if true_labels is not None:
    out_df["Actual"]  = ["Malicious" if l==1 else "Benign" for l in true_labels]
    out_df["Correct"] = (rf_pred == true_labels)

In [ ]:
out_df.to_csv("results/predictions.csv", index=False)

In [ ]:
print(f"\n  Predictions saved  →  results/predictions.csv")

================================================================
FINAL SUMMARY — printed below the code
================================================================

In [ ]:
print("\n" + "=" * 65)
print("  PREDICTION COMPLETE — FULL SUMMARY")
print("=" * 65)
print()
print(f"  Input file            :  {input_file}")
print(f"  Total files scanned   :  {total_files}")
print(f"  Decision threshold    :  {threshold}")
print()
print("  PREDICTION BREAKDOWN")
print(f"    Flagged MALICIOUS   :  {flagged_mal}  ({flagged_mal/total_files*100:.1f}%)")
print(f"    Marked Benign       :  {flagged_benign}  ({flagged_benign/total_files*100:.1f}%)")
print()
print("  RISK LEVELS")
print(f"    Very High Risk  (score >= 0.9)  :  {high_risk} files")
print(f"    Medium Risk     (score 0.5-0.9) :  {medium_risk} files")
print(f"    Low Risk        (score < 0.5)   :  {low_risk} files")
print()
print(f"  Score Statistics")
print(f"    Average score   :  {rf_proba.mean():.4f}")
print(f"    Highest score   :  {rf_proba.max():.4f}")
print(f"    Lowest score    :  {rf_proba.min():.4f}")

In [ ]:
if true_labels is not None:
    correct = (rf_pred == true_labels).sum()
    acc     = correct / len(true_labels) * 100
    tp_val  = int(((rf_pred==1) & (true_labels==1)).sum())
    tn_val  = int(((rf_pred==0) & (true_labels==0)).sum())
    fp_val  = int(((rf_pred==1) & (true_labels==0)).sum())
    fn_val  = int(((rf_pred==0) & (true_labels==1)).sum())
    print()
    print("  ACCURACY  (actual labels available)")
    print(f"    Accuracy          :  {acc:.2f}%")
    print(f"    Correct           :  {correct} out of {len(true_labels)}")
    print(f"    Wrong             :  {len(true_labels)-correct}")
    print()
    print("  CONFUSION MATRIX")
    print(f"    True Positives  (caught malware)    :  {tp_val}")
    print(f"    True Negatives  (correct benign)    :  {tn_val}")
    print(f"    False Positives (false alarms)      :  {fp_val}")
    print(f"    False Negatives (missed malware)    :  {fn_val}  ← most dangerous")

In [ ]:
print()
print(f"  Full results saved  :  results/predictions.csv")
print("=" * 65)